# Импорт модулей

In [2]:
import os
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import ttest_rel
from itertools import product
from sklearn.metrics import adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

from config import PERIODS, METHODS, TRADING_DAYS, SHARPE_THRESHOLD, SORTINO_THRESHOLD
from utils import (
    equal_weight_metrics, ensure_dirs, normalize_pair, normalize_triple, normalize_quad,
    load_prices_dict, get_dividends_sum, compute_sharpe_with_dividends, optimize_portfolio
)

ensure_dirs(['results/tables', 'results/figures'])

# Загрузка общих портфелей из корреляционного подхода

In [4]:
corr_pairs_file = 'results/tables/common_pairs_all_datasets.csv'
corr_triples_file = 'results/tables/common_triples_all_datasets.csv'

if not os.path.exists(corr_pairs_file):
    raise FileNotFoundError(f"{corr_pairs_file} не найден. Сначала выполните 02_correlation_selection.ipynb")
if not os.path.exists(corr_triples_file):
    raise FileNotFoundError(f"{corr_triples_file} не найден. Сначала выполните 02_correlation_selection.ipynb")

pairs_corr = pd.read_csv(corr_pairs_file)
triples_corr = pd.read_csv(corr_triples_file)
print(f"Загружено пар (корреляционный подход): {len(pairs_corr)}")
print(f"Загружено троек (корреляционный подход): {len(triples_corr)}")

Загружено пар (корреляционный подход): 51
Загружено троек (корреляционный подход): 17


# Загрузка общих портфелей из кластерного подхода

In [6]:
cluster_pairs_file = 'results/tables/common_cluster_portfolios_2.csv'
cluster_triples_file = 'results/tables/common_cluster_portfolios_3.csv'

pairs_cluster = pd.read_csv(cluster_pairs_file) if os.path.exists(cluster_pairs_file) else None
triples_cluster = pd.read_csv(cluster_triples_file) if os.path.exists(cluster_triples_file) else None

if pairs_cluster is None:
    print("Внимание: общие пары из кластерного подхода не найдены.")
else:
    print(f"Загружено пар (кластерный подход): {len(pairs_cluster)}")
if triples_cluster is None:
    print("Внимание: общие тройки из кластерного подхода не найдены.")
else:
    print(f"Загружено троек (кластерный подход): {len(triples_cluster)}")

Загружено пар (кластерный подход): 6
Загружено троек (кластерный подход): 1


# Функция обработки всех портфелей для заданного набора данных

In [8]:
def process_portfolios_for_dataset(portfolios_df, n, period_name, method):
    price_file = os.path.join('data', f"sortino_tickers_prices_{period_name}_{method}.csv")
    df_prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
    returns = df_prices.pct_change().dropna()
    mean_ret = returns.mean()
    cov = returns.cov()
    rf = PERIODS[period_name]['rf']
    
    results = []
    for idx, row in portfolios_df.iterrows():
        tickers = [row[f'ticker{i+1}'] for i in range(n)]
        if not all(t in mean_ret.index for t in tickers):
            continue
        opt = optimize_portfolio(tickers, mean_ret, cov, returns, rf)
        if opt is None:
            continue
        record = {f'ticker{i+1}': tickers[i] for i in range(n)}
        record.update({
            'old_return': opt['old_return'],
            'old_risk': opt['old_risk'],
            'old_sharpe': opt['old_sharpe'],
            'new_return': opt['new_return'],
            'new_risk': opt['new_risk'],
            'new_sharpe': opt['new_sharpe'],
            'improved': opt['improved']
        })
        for i, w in enumerate(opt['weights']):
            record[f'weight{i+1}_opt'] = w
        results.append(record)
    return pd.DataFrame(results)

# Оптимизация для всех наборов (корреляционные портфели)

In [10]:
corr_results = {}  

for period_name, method in product(PERIODS.keys(), METHODS):

    df_opt_pairs = process_portfolios_for_dataset(pairs_corr, 2, period_name, method)
    out_file = os.path.join('results/tables', f"optimized_corr_pairs_{period_name}_{method}.csv")
    df_opt_pairs.to_csv(out_file, index=False)
    corr_results[(period_name, method, 2)] = df_opt_pairs
    
    df_opt_triples = process_portfolios_for_dataset(triples_corr, 3, period_name, method)
    out_file = os.path.join('results/tables', f"optimized_corr_triples_{period_name}_{method}.csv")
    df_opt_triples.to_csv(out_file, index=False)
    corr_results[(period_name, method, 3)] = df_opt_triples
    
    print(f"{period_name} {method}: пар улучшено = {df_opt_pairs['improved'].sum()}/{len(df_opt_pairs)}; троек улучшено = {df_opt_triples['improved'].sum()}/{len(df_opt_triples)}")

2023_2024 inner: пар улучшено = 2/51; троек улучшено = 0/17
2023_2024 ffill: пар улучшено = 2/51; троек улучшено = 0/17
2024_2025 inner: пар улучшено = 0/51; троек улучшено = 0/17
2024_2025 ffill: пар улучшено = 0/51; троек улучшено = 0/17


# Оптимизация для всех наборов (кластерные портфели)

In [12]:
cluster_results = {}
if pairs_cluster is not None:
    for period_name, method in product(PERIODS.keys(), METHODS):
        df_opt = process_portfolios_for_dataset(pairs_cluster, 2, period_name, method)
        out_file = os.path.join('results/tables', f"optimized_cluster_pairs_{period_name}_{method}.csv")
        df_opt.to_csv(out_file, index=False)
        cluster_results[(period_name, method, 2)] = df_opt
        print(f"{period_name} {method} (кластер пары): улучшено = {df_opt['improved'].sum()}/{len(df_opt)}")

if triples_cluster is not None:
    for period_name, method in product(PERIODS.keys(), METHODS):
        df_opt = process_portfolios_for_dataset(triples_cluster, 3, period_name, method)
        out_file = os.path.join('results/tables', f"optimized_cluster_triples_{period_name}_{method}.csv")
        df_opt.to_csv(out_file, index=False)
        cluster_results[(period_name, method, 3)] = df_opt
        print(f"{period_name} {method} (кластер тройки): улучшено = {df_opt['improved'].sum()}/{len(df_opt)}")

2023_2024 inner (кластер пары): улучшено = 0/6
2023_2024 ffill (кластер пары): улучшено = 0/6
2024_2025 inner (кластер пары): улучшено = 0/6
2024_2025 ffill (кластер пары): улучшено = 0/6
2023_2024 inner (кластер тройки): улучшено = 0/1
2023_2024 ffill (кластер тройки): улучшено = 0/1
2024_2025 inner (кластер тройки): улучшено = 0/1
2024_2025 ffill (кластер тройки): улучшено = 0/1


# Метрики для сравнения подходов

In [14]:
def aggregate_improvements(results_dict, n, portfolio_type):
    all_improved = []
    all_risk_reduction = []
    all_sharpe_increase = []
    total_portfolios = 0
    
    for (period_name, method, sz), df in results_dict.items():
        if sz != n:
            continue
        if df is None or df.empty:
            continue
        total_portfolios += len(df)
        improved = df[df['improved']]
        all_improved.append(len(improved))
        if len(improved) > 0:
            rel_risk = (improved['old_risk'] - improved['new_risk']) / improved['old_risk']
            all_risk_reduction.extend(rel_risk.tolist())
            sharpe_inc = improved['new_sharpe'] - improved['old_sharpe']
            all_sharpe_increase.extend(sharpe_inc.tolist())
    
    if total_portfolios == 0:
        return None
    total_improved = sum(all_improved)
    pct_improved = total_improved / total_portfolios * 100
    avg_risk_red = np.mean(all_risk_reduction) * 100 if all_risk_reduction else np.nan
    avg_sharpe_inc = np.mean(all_sharpe_increase) if all_sharpe_increase else np.nan
    return {
        'total': total_portfolios,
        'improved': total_improved,
        'pct_improved': pct_improved,
        'avg_risk_reduction_pct': avg_risk_red,
        'avg_sharpe_increase': avg_sharpe_inc
    }

corr_agg_2 = aggregate_improvements(corr_results, 2, 'corr')
corr_agg_3 = aggregate_improvements(corr_results, 3, 'corr')
print("\nКорреляционный подход")
print(f"Пары: {corr_agg_2}")
print(f"Тройки: {corr_agg_3}")

cluster_agg_2 = aggregate_improvements(cluster_results, 2, 'cluster')
cluster_agg_3 = aggregate_improvements(cluster_results, 3, 'cluster')
print("\nКластерный подход")
print(f"Пары: {cluster_agg_2}")
print(f"Тройки: {cluster_agg_3}")


=== Корреляционный подход ===
Пары: {'total': 204, 'improved': 4, 'pct_improved': 1.9607843137254901, 'avg_risk_reduction_pct': 49.56474063846274, 'avg_sharpe_increase': 0.46984706006074206}
Тройки: {'total': 68, 'improved': 0, 'pct_improved': 0.0, 'avg_risk_reduction_pct': nan, 'avg_sharpe_increase': nan}

=== Кластерный подход ===
Пары: {'total': 24, 'improved': 0, 'pct_improved': 0.0, 'avg_risk_reduction_pct': nan, 'avg_sharpe_increase': nan}
Тройки: {'total': 4, 'improved': 0, 'pct_improved': 0.0, 'avg_risk_reduction_pct': nan, 'avg_sharpe_increase': nan}


In [16]:
comparison_data = []
for method_name, agg in [('Корреляционный', corr_agg_2), ('Кластерный', cluster_agg_2)]:
    if agg:
        comparison_data.append({
            'Метод': method_name,
            'Количество портфелей': agg['total'],
            'Доля улучшившихся, %': round(agg['pct_improved'], 1),
            'Среднее снижение риска, %': round(agg['avg_risk_reduction_pct'], 1) if not np.isnan(agg['avg_risk_reduction_pct']) else '-',
            'Среднее увеличение Шарпа': round(agg['avg_sharpe_increase'], 2) if not np.isnan(agg['avg_sharpe_increase']) else '-'
        })
df_comparison = pd.DataFrame(comparison_data)
df_comparison.to_csv('results/tables/final_comparison_pairs.csv', index=False)
print("\nИтоговое сравнение пар")
print(df_comparison.to_string(index=False))


=== Итоговое сравнение (пары) ===
         Метод  Количество портфелей  Доля улучшившихся, % Среднее снижение риска, % Среднее увеличение Шарпа
Корреляционный                   204                   2.0                      49.6                     0.47
    Кластерный                    24                   0.0                         -                        -
